In [1]:
!pip install -q transformers datasets torch scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch
import os

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
SAVE_PATH="/content/drive/MyDrive/VoiceVision-AI/06_Intent_Model"

os.makedirs(SAVE_PATH,exist_ok=True)

print("Folder Created")

Folder Created


In [5]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/VoiceVision-AI/01_Datasets/Waste_Images/intent_dataset.csv"

df = pd.read_csv(CSV_PATH)
df.head()

,text,intent
0,Is this mouse biodegradable?,Decomposition
1,Is there a special bin for magazine disposal?,Disposal
2,Is this cardboard dangerous for animals?,Environment
3,How many years does it take for fabric to vanish?,Decomposition
4,Where should I throw this mobile charger?,Disposal


In [6]:
print(df.shape)
print(df.columns)
df.head()

(600, 2)
Index(['text', 'intent'], dtype='object')


,text,intent
0,Is this mouse biodegradable?,Decomposition
1,Is there a special bin for magazine disposal?,Disposal
2,Is this cardboard dangerous for animals?,Environment
3,How many years does it take for fabric to vanish?,Decomposition
4,Where should I throw this mobile charger?,Disposal


In [7]:
print(df.isnull().sum())

text      0
intent    0
dtype: int64


In [8]:
print(df["intent"].value_counts())

intent
Decomposition    100
Disposal         100
Environment      100
Reuse            100
Recycling        100
Material         100
Name: count, dtype: int64


In [9]:
print("Before:", len(df))

df = df.drop_duplicates()

print("After :", len(df))

Before: 600
After : 600


In [10]:
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

df.head()

,text,intent
0,"I'm thinking of reusing this aluminum can, is ...",Reuse
1,How should I throw away this keyboard properly?,Disposal
2,Does this newspaper pose a risk to the ecosystem?,Environment
3,Is decomposition slow for steel can like this?,Decomposition
4,Should I donate this egg carton so someone els...,Reuse


In [11]:
labels = sorted(df["intent"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

df["label"] = df["intent"].map(label2id)

df.head()

,text,intent,label
0,"I'm thinking of reusing this aluminum can, is ...",Reuse,5
1,How should I throw away this keyboard properly?,Disposal,1
2,Does this newspaper pose a risk to the ecosystem?,Environment,2
3,Is decomposition slow for steel can like this?,Decomposition,0
4,Should I donate this egg carton so someone els...,Reuse,5


In [12]:
import json
import os

SAVE_PATH = "/content/drive/MyDrive/VoiceVision-AI/06_Intent_Model"

os.makedirs(SAVE_PATH, exist_ok=True)

with open(os.path.join(SAVE_PATH, "label_mapping.json"), "w") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        indent=4
    )

print("Label mapping saved successfully.")

Label mapping saved successfully.


In [13]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Training Samples :", len(train_df))
print("Testing Samples  :", len(test_df))

Training Samples : 480
Testing Samples  : 120


In [14]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'intent', 'label', '__index_level_0__'],
    num_rows: 480
})
Dataset({
    features: ['text', 'intent', 'label', '__index_level_0__'],
    num_rows: 120
})


In [15]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [16]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

In [17]:
print(train_dataset)

Dataset({
    features: ['text', 'intent', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 480
})


In [18]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

print("Dataset converted to PyTorch tensors successfully!")

Dataset converted to PyTorch tensors successfully!


In [19]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

print("DistilBERT Loaded Successfully!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT Loaded Successfully!


In [20]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

In [21]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics

)

In [23]:
import torch
import torchvision
import transformers
import datasets

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
Transformers: 5.12.1
Datasets: 4.0.0
CUDA: True


In [24]:
import torch

print("GPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name :", torch.cuda.get_device_name(0))

GPU Available : True
GPU Name : Tesla T4


In [26]:
import sys

# Remove torchvision from loaded modules
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]

print("torchvision removed from memory")

torchvision removed from memory


In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.548654,1.366861,0.900000
2,0.933566,0.727824,0.983333
3,0.484298,0.359717,0.991667
4,0.295242,0.221818,1.000000
5,0.222983,0.187363,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.785214942296346, metrics={'train_runtime': 43.8376, 'train_samples_per_second': 54.747, 'train_steps_per_second': 3.422, 'total_flos': 39743054438400.0, 'train_loss': 0.785214942296346, 'epoch': 5.0})

In [28]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.222983,0.221818,5,1.000000


{'eval_loss': 0.2218184620141983, 'eval_accuracy': 1.0}


In [29]:
trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Intent Detection Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Intent Detection Model Saved Successfully!


In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/VoiceVision-AI/06_Intent_Model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Model Loaded Successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model Loaded Successfully!


In [31]:
import torch

question = "Can I recycle this bottle?"

inputs = tokenizer(
    question,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Question :", question)
print("Predicted Intent :", id2label[prediction])

Question : Can I recycle this bottle?
Predicted Intent : Recycling


In [32]:
questions = [
    "Where should I throw this bottle?",
    "Can I recycle this?",
    "Can I reuse this container?",
    "What material is this made of?",
    "Is this harmful to the environment?",
    "How long does this take to decompose?"
]

for q in questions:

    inputs = tokenizer(
        q,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    print("=" * 60)
    print("Question :", q)
    print("Predicted Intent :", id2label[prediction])

Question : Where should I throw this bottle?
Predicted Intent : Disposal
Question : Can I recycle this?
Predicted Intent : Recycling
Question : Can I reuse this container?
Predicted Intent : Reuse
Question : What material is this made of?
Predicted Intent : Material
Question : Is this harmful to the environment?
Predicted Intent : Environment
Question : How long does this take to decompose?
Predicted Intent : Decomposition


In [33]:
new_questions = [
    "Can I put this in the recycling bin?",
    "Which dustbin should I use for this?",
    "Will this item damage nature?",
    "How many years before this breaks down?",
    "Can I use this bottle again?",
    "Is this made from plastic?"
]

for q in new_questions:

    inputs = tokenizer(
        q,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    print("=" * 60)
    print("Question :", q)
    print("Predicted Intent :", id2label[prediction])

Question : Can I put this in the recycling bin?
Predicted Intent : Recycling
Question : Which dustbin should I use for this?
Predicted Intent : Disposal
Question : Will this item damage nature?
Predicted Intent : Environment
Question : How many years before this breaks down?
Predicted Intent : Decomposition
Question : Can I use this bottle again?
Predicted Intent : Reuse
Question : Is this made from plastic?
Predicted Intent : Material
